In [2]:
#SMOTE BASED AUGMENTATION:

In [8]:
import os
import cv2
import gc
import math
import random
import time
import shutil
import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image
from pathlib import Path
from glob import glob 
import matplotlib.pyplot as plt 
import tensorflow as tf
from collections import defaultdict
from imblearn.over_sampling import SMOTE
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split 
from tensorflow.keras.preprocessing.image import img_to_array, array_to_img, load_img, save_img, ImageDataGenerator

imagegen = ImageDataGenerator()

In [9]:
base_path = "D:/final year project/Second Set/"
balanced_path = "D:/final year project/smote_augmented/balanced_dataset/"
preprocessed_path1 = "D:/final year project/smote_augmented/denoised_dataset/"
preprocessed_path2 = "D:/final year project/smote_augmented/normalized_dataset/"
preprocessed_path3 = "D:/final year project/smote_augmented/enhanced_dataset/"
augmented_dataset = "D:/final year project/smote_augmented/augmented_dataset/"
os.makedirs(balanced_path, exist_ok=True)
os.makedirs(preprocessed_path1, exist_ok=True)
os.makedirs(preprocessed_path2, exist_ok=True)
os.makedirs(preprocessed_path3, exist_ok=True)
os.makedirs(augmented_dataset, exist_ok=True)

In [6]:
class ImageSMOTE:
    def __init__(self, input_folder, output_folder, desired_samples_per_class=None, k_neighbors=5, target_size=(64, 64)):
        """
        Initialize the ImageSMOTE class with memory-efficient settings.      
        Args:
            input_folder (str): Path to the input folder containing class subfolders
            output_folder (str): Path to save the augmented dataset
            desired_samples_per_class (int): Number of desired samples per class
            k_neighbors (int): Number of nearest neighbors for SMOTE
            target_size (tuple): Target size for resizing images (height, width)
        """
        
        self.input_folder = input_folder
        self.output_folder = output_folder
        self.desired_samples = desired_samples_per_class
        self.k_neighbors = k_neighbors
        self.target_size = target_size
        self.image_extensions = ('.png', '.jpg')
        
        os.makedirs(self.output_folder, exist_ok=True)
    
    def load_images(self):
        class_folders = [d for d in os.listdir(self.input_folder) 
                         if os.path.isdir(os.path.join(self.input_folder, d))]
        
        self.image_paths = defaultdict(list)
        self.image_data = defaultdict(list)
        self.class_names = []
        
        print("Loading and resizing images>>>")
        for class_name in tqdm(class_folders):
            class_path = os.path.join(self.input_folder, class_name)
            image_files = [f for f in os.listdir(class_path) 
                          if f.lower().endswith(self.image_extensions)]
            
            self.class_names.append(class_name)
            for img_file in image_files:
                img_path = os.path.join(class_path, img_file)
                self.image_paths[class_name].append(img_path)
               
                img = cv2.imread(img_path)
                if img is None:
                    print("There is no image found on that path")
                    continue  
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, self.target_size)
                self.image_data[class_name].append(img)
        
        return self.image_data
    
    def preprocess_images(self, image_data):
        processed_data = {}
        
        print("Preprocessing images>>>")
        for class_name, images in tqdm(image_data.items()):
            # Process images in batches to save memory
            batch_size = 50
            processed_vectors = []
            
            for i in range(0, len(images), batch_size):
                batch = images[i:i+batch_size] 
                #Normalize
                vectors = [img.astype(np.float32) / 255.0 for img in batch]
                vectors = [v.flatten() for v in vectors]
                processed_vectors.extend(vectors)
                
                del batch, vectors
                gc.collect()
            
            processed_data[class_name] = np.array(processed_vectors, dtype=np.float32)
        
        return processed_data
    
    def apply_smote(self, processed_data):
        """Apply SMOTE with memory-efficient implementation."""
        class_counts = {k: len(v) for k, v in processed_data.items()}
        max_count = max(class_counts.values())
        
        if self.desired_samples is None:
            self.desired_samples = max_count
        else:
            self.desired_samples = min(self.desired_samples, max_count)
        
        print(f"Target samples per class: {self.desired_samples}")
        
        augmented_data = {}
        
        print("Applying SMOTE...")
        for class_name in tqdm(self.class_names):
            current_data = processed_data[class_name]
            current_count = current_data.shape[0]
            
            if current_count >= self.desired_samples:
                indices = np.random.choice(current_count, self.desired_samples, replace=False)
                augmented_data[class_name] = current_data[indices]
                continue
            
            n_neighbors = min(self.k_neighbors, current_count - 1)
            n_synthetic = self.desired_samples - current_count
            
            # Generate synthetic samples with memory management
            synthetic_samples = self._generate_synthetic_samples_mem(
                current_data, 
                n_synthetic, 
                n_neighbors=n_neighbors,
                batch_size=50
            )
            
            augmented_data[class_name] = np.vstack([current_data, synthetic_samples])
            
            del synthetic_samples
            gc.collect()
        
        return augmented_data
    
    def _generate_synthetic_samples_mem(self, samples, n_synthetic, n_neighbors=5, batch_size=50):
        synthetic_samples = []
        
        # Process in smaller chunks to avoid memory overload
        for chunk_start in range(0, len(samples), 100):
            chunk_end = min(chunk_start + 100, len(samples))
            chunk = samples[chunk_start:chunk_end]
            
            knn = NearestNeighbors(n_neighbors=n_neighbors)
            knn.fit(chunk)
            
            for _ in range(0, n_synthetic, batch_size):
                current_batch_size = min(batch_size, n_synthetic - len(synthetic_samples))
                batch = []
                
                for __ in range(current_batch_size):
                    idx = np.random.randint(0, len(chunk))
                    sample = chunk[idx]
                    _, indices = knn.kneighbors([sample])
                    neighbor_idx = np.random.choice(indices[0])
                    neighbor = chunk[neighbor_idx]
                    alpha = np.random.random()
                    synthetic = sample + alpha * (neighbor - sample)
                    batch.append(synthetic)
                
                synthetic_samples.extend(batch)
                
                del batch
                gc.collect()
        
        return np.array(synthetic_samples, dtype=np.float32)
    
    def reconstruct_images(self, augmented_data): 
        reconstructed_data = {}
        original_shape = (*self.target_size, 3)  # Height, Width, Channels
        
        print("Reconstructing images in batches...")
        for class_name, vectors in tqdm(augmented_data.items()):
            images = []
            batch_size = 50
            
            for i in range(0, len(vectors), batch_size):
                batch = vectors[i:i+batch_size]
                for vec in batch:
                    img = (vec.reshape(original_shape) * 255)
                    img = np.clip(img, 0, 255).astype(np.uint8)
                    images.append(img)
                
                del batch
                gc.collect()
            
            reconstructed_data[class_name] = images
        
        return reconstructed_data
    
    def save_images(self, reconstructed_data): 
        print("Saving augmented images...")
        for class_name, images in tqdm(reconstructed_data.items()):
            class_path = os.path.join(self.output_folder, class_name)
            os.makedirs(class_path, exist_ok=True)
            
            existing_files = [f for f in os.listdir(class_path) 
                            if f.lower().endswith(self.image_extensions)]
            start_idx = len(existing_files)
            
            batch_size = 20
            for i in range(0, len(images), batch_size):
                batch = images[i:i+batch_size]
                for j, img in enumerate(batch):
                    img_path = os.path.join(class_path, f"{class_name}_aug_{start_idx + i + j}.png")
                    pil_img = Image.fromarray(img)
                    pil_img.save(img_path)
                
                # Clear memory between batches
                del batch
                gc.collect()
    
    def run(self): 
        try:
            # Step 1: Load images
            image_data = self.load_images()
            
            # Step 2: Preprocess images
            processed_data = self.preprocess_images(image_data)
            del image_data  # Free memory
            gc.collect()
            
            # Step 3: Apply SMOTE
            augmented_data = self.apply_smote(processed_data)
            del processed_data  # Free memory
            gc.collect()
            
            # Step 4: Reconstruct images
            reconstructed_data = self.reconstruct_images(augmented_data)
            del augmented_data  # Free memory
            gc.collect()
            
            # Step 5: Save images
            self.save_images(reconstructed_data)
            
            print("SMOTE augmentation completed successfully!")
        except Exception as e:
            print(f"Error during SMOTE augmentation: {str(e)}")
            raise


if __name__ == "__main__":     
    smote = ImageSMOTE(
        input_folder=base_path,
        output_folder=balanced_path,
        desired_samples_per_class=None, 
        k_neighbors=5,
        target_size=(256, 256) 
    )
    smote.run()

Loading and resizing images>>>


  0%|                                                                                            | 0/2 [00:05<?, ?it/s]


KeyboardInterrupt: 

In [15]:
#DENOISING:

# Wiener filter for a single channel 
def wiener_filter(img, kernel_size=3, noise_power=None):
    img = tf.cast(img, tf.float32)
    kernel = tf.ones((kernel_size, kernel_size, 1, 1)) / (kernel_size * kernel_size)
    mean = tf.nn.convolution(tf.expand_dims(img, axis=0), kernel, padding='SAME')
    mean = tf.squeeze(mean, axis=0)

    sqr_mean = tf.nn.convolution(tf.expand_dims(tf.square(img), axis=0), kernel, padding='SAME')
    sqr_mean = tf.squeeze(sqr_mean, axis=0)
    variance = sqr_mean - tf.square(mean)
    variance = tf.clip_by_value(variance, 0.0, tf.float32.max)

    gain = variance / (variance + noise_power)
    filtered_img = mean + gain * (img - mean)
    return filtered_img

# Apply Wiener filter channel-wise for color images
def wiener_filter_color(image, kernel_size=3, noise_power=None):
    channels = tf.unstack(image, axis=-1)
    filtered_channels = [
        # Squeeze the channel dimension after filtering
        tf.squeeze(wiener_filter(tf.expand_dims(c, axis=-1), kernel_size, noise_power), axis=-1)
        for c in channels
    ]
    return tf.stack(filtered_channels, axis=-1)

In [16]:
#Check noise median: 
def estimate_noise_mad_rgb(img_path):
    img = cv2.imread(img_path, cv2.IMREAD_COLOR).astype(np.float32) / 255.0
    noise_vars = []
    for c in range(3): 
        channel = img[..., c]
        grad = cv2.Laplacian(channel, cv2.CV_32F)
        sigma = np.median(np.abs(grad - np.median(grad))) / 0.6745
        noise_vars.append(sigma ** 2)
    return np.mean(noise_vars)

#Denoise function:
def denoise_dataset(input_dir, output_dir, kernel_size=3):
 
    os.makedirs(output_dir, exist_ok=True)
    
    # Get all class folders
    class_folders = [d for d in os.listdir(input_dir) if os.path.isdir(os.path.join(input_dir, d))]
    
    for class_name in tqdm(class_folders, desc="Processing classes"):
        class_input_dir = os.path.join(input_dir, class_name)
        class_output_dir = os.path.join(output_dir, class_name)
        os.makedirs(class_output_dir, exist_ok=True)
        
        # Get all images in class folder
        image_files = [f for f in os.listdir(class_input_dir) 
                      if f.lower().endswith(('.jpg', '.png'))]
        
        for img_file in tqdm(image_files, desc=f"Processing {class_name}", leave=False):
            img_path = os.path.join(class_input_dir, img_file)
            
            # Load and normalize image
            img = cv2.imread(img_path)
            if img is None:
                print(f"Warning: Could not read image {img_path}")
                continue

             # Estimate noise for this image
            noise_var = estimate_noise_mad_rgb(img_path)
            # print(f"Noise variance (MAD): {noise_var:.6f}")
            img = img.astype(np.float32) / 255.0
            img_tensor = tf.convert_to_tensor(img)
            
            # Apply Wiener filter
            filtered = wiener_filter_color(img_tensor, kernel_size=kernel_size, noise_power=noise_var)
            filtered_img = (filtered.numpy() * 255).astype(np.uint8)
            
            # Save result
            output_path = os.path.join(class_output_dir, img_file)
            cv2.imwrite(output_path, filtered_img)

if __name__ == "__main__":
    denoise_dataset(balanced_path, preprocessed_path1)

Processing classes:   0%|                                                                        | 0/2 [00:00<?, ?it/s]
Processing 400x Normal Oral Cavity Histopathological Images:   0%|                             | 0/495 [00:00<?, ?it/s]
Processing 400x Normal Oral Cavity Histopathological Images:   0%|                     | 1/495 [00:02<17:33,  2.13s/it]
Processing 400x Normal Oral Cavity Histopathological Images:   0%|                     | 2/495 [00:02<07:50,  1.05it/s]
Processing 400x Normal Oral Cavity Histopathological Images:   1%|▏                    | 3/495 [00:02<04:59,  1.64it/s]
Processing 400x Normal Oral Cavity Histopathological Images:   1%|▏                    | 4/495 [00:02<03:24,  2.40it/s]
Processing 400x Normal Oral Cavity Histopathological Images:   1%|▎                    | 6/495 [00:02<02:06,  3.87it/s]
Processing 400x Normal Oral Cavity Histopathological Images:   2%|▎                    | 8/495 [00:02<01:29,  5.43it/s]
Processing 400x Normal Oral Cavity Histo

In [18]:
#Stain Normalization:

# Optical Density (OD) normalization constants
Io = 240        # Light intensity used in image formation
alpha = 1       # Percentile threshold to exclude extreme angles in stain vectors
beta = 0.15     # Threshold for OD filtering


# Reference H&E stain matrix (predefined based on a reference image)
HERef = tf.constant([[0.5626, 0.2159],
                     [0.7201, 0.8012],
                     [0.4062, 0.5581]], dtype=tf.float32) 

maxCRef = np.array([1.9705, 1.0308])   # Reference stain concentration percentiles


def process_single_image(img):
    h, w, _ = img.shape   # Get image dimensions (height, width, channels)
    img_flat = img.reshape((-1, 3))  # Flatten image to 2D array of pixels (N, 3)

    OD = -tf.math.log((tf.cast(img_flat, tf.float32) + 1) / Io) # Convert RGB to optical density (OD)

    mask = tf.reduce_all(OD >= beta, axis=1) # Mask to filter out low OD values (background or noise)
    
    ODhat = tf.boolean_mask(OD, mask) # Apply mask to retain only meaningful stain regions

    # Perform PCA on ODhat to find principal stain directions
    eigvals, eigvecs = np.linalg.eigh(np.cov(ODhat.numpy().T))
    eigvecs_tf = tf.convert_to_tensor(eigvecs, dtype=tf.float32)

    
    That = tf.matmul(ODhat, eigvecs_tf[:, 1:3]) # Project ODhat onto 2 principal components

    
    phi = tf.math.atan2(That[:, 1], That[:, 0]).numpy() # Compute angles (phi) of projected data for stain vector estimation

    # Compute angle percentiles to exclude outliers
    minPhi = np.percentile(phi, alpha)
    maxPhi = np.percentile(phi, 100 - alpha)

    # Compute stain vectors based on min and max angle projections
    vMin = eigvecs_tf[:, 1:3] @ tf.constant([[np.cos(minPhi)], [np.sin(minPhi)]], dtype=tf.float32)
    vMax = eigvecs_tf[:, 1:3] @ tf.constant([[np.cos(maxPhi)], [np.sin(maxPhi)]], dtype=tf.float32)

    # Order stain vectors: Hematoxylin first, Eosin second
    HE = tf.concat([vMin, vMax], axis=1) if vMin[0] > vMax[0] else tf.concat([vMax, vMin], axis=1)
    
    # Solve OD = HE * C using least squares to get stain concentrations
    Y = tf.transpose(OD)
    C = tf.linalg.lstsq(HE, Y, l2_regularizer=1e-10)

    
    C_np = C.numpy() # Convert concentrations to numpy for further processing

    # Normalize stain concentrations using 99th percentile
    maxC = np.array([np.percentile(C_np[0, :], 99), np.percentile(C_np[1, :], 99)])
    tmp = maxC / maxCRef
    C2 = C_np / tmp[:, np.newaxis]
    C2_tf = tf.convert_to_tensor(C2, dtype=tf.float32)

    # Reconstruct normalized image using reference stain matrix
    Inorm = Io * tf.exp(-tf.matmul(HERef, C2_tf))
    Inorm = tf.clip_by_value(Inorm, 0, 255)

    # Reshape and convert back to image format (uint8)
    Inorm_img = tf.reshape(tf.transpose(Inorm), (h, w, 3)).numpy().astype(np.uint8)
    
    return Inorm_img


def process_directory(input_dir, output_dir):
    class_folders = [d for d in os.listdir(input_dir) if os.path.isdir(os.path.join(input_dir, d))]
    for class_name in class_folders:
        input_class_path = os.path.join(input_dir, class_name)
        output_class_path = os.path.join(output_dir, class_name)
        os.makedirs(output_class_path, exist_ok=True)

        for file in tqdm(os.listdir(input_class_path), desc=f"Processing {class_name}"):
            if file.lower().endswith(('.jpg', '.png')):
                in_path = os.path.join(input_class_path, file)
                try:
                    img = Image.open(in_path).convert("RGB") # Load image and convert to RGB numpy array
                    img_np = np.array(img)
                    
                    norm_img = process_single_image(img_np)   # Perform stain normalization

                    # Save normalized image to output directory
                    out_path = os.path.join(output_class_path, os.path.splitext(file)[0] + "_normalized.png")
                    Image.fromarray(norm_img).save(out_path)
                    
                except Exception as e:
                    print(f"❌ Failed to process {in_path}: {e}")


process_directory(preprocessed_path1, preprocessed_path2)


Processing 400x Normal Oral Cavity Histopathological Images: 100%|███████████████████| 495/495 [00:32<00:00, 15.09it/s]
Processing 400x OSCC Histopathological Images: 100%|█████████████████████████████████| 495/495 [00:31<00:00, 15.68it/s]


In [21]:
#CLAHE

# Create output directory structure
class_dirs = [d for d in os.listdir(preprocessed_path2) if os.path.isdir(os.path.join(preprocessed_path2, d))]
for class_dir in class_dirs:
    os.makedirs(os.path.join(preprocessed_path3, class_dir), exist_ok=True)

# Apply CLAHE using OpenCV
def apply_clahe_to_image(image_np):
    lab = cv2.cvtColor(image_np, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(16, 16))
    l_clahe = clahe.apply(l)
    lab_clahe = cv2.merge((l_clahe, a, b))
    img_clahe = cv2.cvtColor(lab_clahe, cv2.COLOR_LAB2RGB)
    return img_clahe

# Process all images
for class_dir in class_dirs:
    input_class_path = os.path.join(preprocessed_path2, class_dir)
    output_class_path = os.path.join(preprocessed_path3, class_dir)

    image_paths = glob(os.path.join(input_class_path, "*.png"))

    for image_path in tqdm(image_paths, desc=f"Processing {class_dir}"):
 
        image = tf.io.read_file(image_path)
        image = tf.image.decode_png(image, channels=3) 
        image = tf.image.convert_image_dtype(image, tf.uint8)

        image_np = image.numpy()
        
        # Apply CLAHE
        clahe_image = apply_clahe_to_image(image_np)

        # Save output
        image_name = os.path.basename(image_path)
        output_path = os.path.join(output_class_path, image_name)
        cv2.imwrite(output_path, cv2.cvtColor(clahe_image, cv2.COLOR_RGB2BGR)) 

Processing 400x Normal Oral Cavity Histopathological Images: 100%|███████████████████| 495/495 [00:17<00:00, 27.72it/s]
Processing 400x OSCC Histopathological Images: 100%|█████████████████████████████████| 495/495 [00:17<00:00, 28.79it/s]


In [13]:
#DATA AUGMENTATION:

#rotation, shifting, shearing, zoooming, flipping, brightness, contrasting, noise, blurring.

# Files
INPUT_DIR = preprocessed_path3
OUTPUT_DIR = augmented_dataset

# Output classes:
os.makedirs(os.path.join(OUTPUT_DIR, "400x Normal Oral Cavity Histopathological Images"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "400x OSCC Histopathological Images"), exist_ok=True)

#Configurations:
TARGET_COUNT = 5000
BATCH_SIZE = 1

datagen = ImageDataGenerator(
    rotation_range=12,
    width_shift_range=0.07,
    height_shift_range=0.07,
    shear_range=0.05,
    zoom_range=0.1,
    horizontal_flip=True,
    vertical_flip=True,
    fill_mode='reflect',
    brightness_range=[0.8, 1], 
)

# Noise, Blur, Random Contrast
def add_more(image):
    # Normalizing and Gaussian noise 
    image = image.astype(np.float32) / 255.0 
    noise = np.random.normal(loc=0, scale=0.01, size=image.shape).astype(np.float32)
    noisy_image = image + noise
    np.clip(noisy_image, 0.0, 1.0, out=noisy_image)

    # Convert to uint8
    image_uint8 = (noisy_image * 255).astype(np.uint8)

    # Gaussian Blur
    kernel_size = np.random.choice([3, 5])
    blurred_image = cv2.GaussianBlur(image_uint8, (kernel_size, kernel_size), 
                                   sigmaX=np.random.uniform(0.3, 0.5))

    # Random Contrast
    alpha = np.random.uniform(0.7, 1.3)  
    contrast_img = cv2.convertScaleAbs(blurred_image, alpha=alpha, beta=np.random.uniform(-10, 10))

    return contrast_img

# Augment Each Class 
def augment_class(class_name, target_count):
    input_path = os.path.join(INPUT_DIR, class_name)
    output_path = os.path.join(OUTPUT_DIR, class_name)

    os.makedirs(output_path, exist_ok=True)
    
    original_images = [os.path.join(input_path, f) for f in os.listdir(input_path) 
                     if f.endswith(('.jpg', '.png'))]
    existing_augmented = [f for f in os.listdir(output_path) if f.endswith(('.jpg', '.png'))]

    current_count = len(existing_augmented)
    remaining = target_count - current_count

    if remaining <= 0:
        print(f"[INFO] {class_name}: Already has {current_count} images.")
        return

    print(f"\n[INFO] Augmenting '{class_name}' from {current_count} to {target_count} images")

    img_index = 0
    pbar = tqdm(total=remaining, desc=f"Augmenting {class_name}")

    while remaining > 0:
        img_path = original_images[img_index % len(original_images)]

        try:
            with Image.open(img_path).convert('RGB') as img:
                img_array = np.expand_dims(np.array(img), axis=0).astype(np.float32)
                
                # Generate multiple augmentations per image
                num_augmentations = min(remaining, np.random.randint(4, 7))  #4 to 7 images/image
                
                for i, batch in enumerate(datagen.flow(img_array, batch_size=1)):
                    if i >= num_augmentations:
                        break
                        
                    augmented_img = batch[0]
                    augmented = add_more(augmented_img)
                    augmented = np.uint8(augmented)

                    filename = f"aug_{os.path.splitext(os.path.basename(img_path))[0]}_{i}_{int(time.time())}.jpg"
                    save_path = os.path.join(output_path, filename)
                    Image.fromarray(augmented).save(save_path)

                    remaining -= 1
                    pbar.update(1)
                    
                    if remaining <= 0:
                        break

        except Exception as e:
            print(f"[ERROR] Failed on {img_path}: {str(e)}")

        img_index += 1

    pbar.close()
    print(f"[DONE] {class_name}: Total images = {len(os.listdir(output_path))}")

# Run Augmentation
augment_class("400x Normal Oral Cavity Histopathological Images", TARGET_COUNT)
augment_class("400x OSCC Histopathological Images", TARGET_COUNT)

# Statistics:
print(f"\n[SUCCESS] Augmentation complete. Output saved to: {OUTPUT_DIR}")
print(f"Final counts:")
print(f"- Normal: {len(os.listdir(os.path.join(OUTPUT_DIR, '400x Normal Oral Cavity Histopathological Images')))} images")
print(f"- OSCC: {len(os.listdir(os.path.join(OUTPUT_DIR, '400x OSCC Histopathological Images')))} images")


[INFO] Augmenting '400x Normal Oral Cavity Histopathological Images' from 0 to 5000 images


Augmenting 400x Normal Oral Cavity Histopathological Images: 100%|█████████████████| 5000/5000 [07:05<00:00, 11.76it/s]


[DONE] 400x Normal Oral Cavity Histopathological Images: Total images = 5000

[INFO] Augmenting '400x OSCC Histopathological Images' from 0 to 5000 images


Augmenting 400x OSCC Histopathological Images: 100%|███████████████████████████████| 5000/5000 [06:18<00:00, 13.20it/s]


[DONE] 400x OSCC Histopathological Images: Total images = 5000

[SUCCESS] Augmentation complete. Output saved to: D:/final year project/smote_augmented/augmented_dataset/
Final counts:
- Normal: 5000 images
- OSCC: 5000 images
